# HW1: Song Lyrics Genre Classification
CSE572 - Tree-based models for classifying song lyrics into 14 genres.

In [ ]:
# Cell 1: Imports & Setup
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)

In [ ]:
# Cell 2: Load Data
train_df = pd.read_csv('song-classification-hw1/train.csv')
test_df = pd.read_csv('song-classification-hw1/test.csv')
sample_sub = pd.read_csv('song-classification-hw1/sample_submission.csv')

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Columns:', list(train_df.columns))
print('\nGenre distribution:')
print(train_df['genre'].value_counts())
print('\nMulti-label songs:', train_df['genre'].str.contains(',').sum())
train_df.head()

## Part 1: Preprocessing (20pt)

In [ ]:
# Cell 3: Preprocessing

# --- Multi-label handling: duplicate rows for each genre ---
def explode_genres(df):
    """Split multi-label genres into separate rows."""
    df = df.copy()
    df['genre'] = df['genre'].str.split(',')
    df = df.explode('genre')
    df['genre'] = df['genre'].str.strip()
    return df.reset_index(drop=True)

train_exp = explode_genres(train_df)
print(f'After exploding multi-labels: {len(train_df)} -> {len(train_exp)} rows')
print('\nGenre distribution after exploding:')
print(train_exp['genre'].value_counts())

# --- Text preprocessing ---
SECTION_HEADERS = re.compile(
    r'\b(intro|verse\s*\d*|chorus|pre[- ]?chorus|bridge|outro|hook|refrain|interlude|instrumental|post[- ]?chorus|breakdown|skit|letra\s+de\s+\w+)\b',
    re.IGNORECASE
)

def clean_lyrics(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    # Remove section headers
    text = SECTION_HEADERS.sub(' ', text)
    # Remove punctuation and special characters
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_exp['lyrics_processed'] = train_exp['lyrics_clean'].apply(clean_lyrics)
test_df['lyrics_processed'] = test_df['lyrics_clean'].apply(clean_lyrics)

# --- Feature engineering ---
def add_meta_features(df):
    df = df.copy()
    words = df['lyrics_processed'].str.split()
    df['word_count'] = words.apply(len)
    df['unique_word_ratio'] = words.apply(lambda w: len(set(w)) / max(len(w), 1))
    df['avg_word_length'] = words.apply(lambda w: np.mean([len(x) for x in w]) if len(w) > 0 else 0)
    return df

train_exp = add_meta_features(train_exp)
test_df = add_meta_features(test_df)

print('\nSample processed lyrics (first 100 chars):')
print(train_exp['lyrics_processed'].iloc[0][:100])

In [ ]:
# Cell 4: Feature Matrix Construction

# TF-IDF on lyrics
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english',
    min_df=2,
    max_df=0.95
)

X_tfidf_train = tfidf.fit_transform(train_exp['lyrics_processed'])
X_tfidf_test = tfidf.transform(test_df['lyrics_processed'])

# Language one-hot encoding
all_languages = pd.concat([train_exp['language'], test_df['language']]).unique()
lang_train = pd.get_dummies(train_exp['language']).reindex(columns=all_languages, fill_value=0)
lang_test = pd.get_dummies(test_df['language']).reindex(columns=all_languages, fill_value=0)

# Metadata features
meta_cols = ['word_count', 'unique_word_ratio', 'avg_word_length']
meta_train = train_exp[meta_cols].values
meta_test = test_df[meta_cols].values

# Combine all features
from scipy.sparse import csr_matrix
X_train = hstack([X_tfidf_train, csr_matrix(lang_train.values), csr_matrix(meta_train)])
X_test = hstack([X_tfidf_test, csr_matrix(lang_test.values), csr_matrix(meta_test)])

# Encode labels
le = LabelEncoder()
y_train = le.fit_transform(train_exp['genre'])

print(f'Feature matrix: {X_train.shape}')
print(f'Classes: {le.classes_}')
print(f'Test matrix: {X_test.shape}')

## Part 2: Random Forest Evaluation (20pt)

In [ ]:
# Cell 5: n_estimators experiment
n_estimators_values = [10, 50, 100, 150, 200, 300]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

n_est_results = []
for n in n_estimators_values:
    rf = RandomForestClassifier(n_estimators=n, random_state=SEED, n_jobs=-1)
    scores = cross_validate(rf, X_train, y_train, cv=cv,
                           scoring='accuracy', return_train_score=True)
    n_est_results.append({
        'n_estimators': n,
        'train_mean': scores['train_score'].mean(),
        'train_std': scores['train_score'].std(),
        'val_mean': scores['test_score'].mean(),
        'val_std': scores['test_score'].std()
    })
    print(f'n_estimators={n:>3d}: train={scores["train_score"].mean():.4f}±{scores["train_score"].std():.4f}, '
          f'val={scores["test_score"].mean():.4f}±{scores["test_score"].std():.4f}')

n_est_df = pd.DataFrame(n_est_results)
n_est_df

In [ ]:
# Cell 6: n_estimators plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(n_est_df['n_estimators'], n_est_df['train_mean'], yerr=n_est_df['train_std'],
            marker='o', label='Training Accuracy', capsize=3)
ax.errorbar(n_est_df['n_estimators'], n_est_df['val_mean'], yerr=n_est_df['val_std'],
            marker='s', label='Validation Accuracy', capsize=3)
ax.set_xlabel('n_estimators')
ax.set_ylabel('Accuracy')
ax.set_title('Random Forest: Effect of n_estimators')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: min_samples_leaf experiment
min_leaf_values = [1, 2, 5, 10, 20, 50]

leaf_results = []
for m in min_leaf_values:
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=m, random_state=SEED, n_jobs=-1)
    scores = cross_validate(rf, X_train, y_train, cv=cv,
                           scoring='accuracy', return_train_score=True)
    leaf_results.append({
        'min_samples_leaf': m,
        'train_mean': scores['train_score'].mean(),
        'train_std': scores['train_score'].std(),
        'val_mean': scores['test_score'].mean(),
        'val_std': scores['test_score'].std()
    })
    print(f'min_samples_leaf={m:>2d}: train={scores["train_score"].mean():.4f}±{scores["train_score"].std():.4f}, '
          f'val={scores["test_score"].mean():.4f}±{scores["test_score"].std():.4f}')

leaf_df = pd.DataFrame(leaf_results)
leaf_df

In [ ]:
# Cell 8: min_samples_leaf plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(leaf_df['min_samples_leaf'], leaf_df['train_mean'], yerr=leaf_df['train_std'],
            marker='o', label='Training Accuracy', capsize=3)
ax.errorbar(leaf_df['min_samples_leaf'], leaf_df['val_mean'], yerr=leaf_df['val_std'],
            marker='s', label='Validation Accuracy', capsize=3)
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('Accuracy')
ax.set_title('Random Forest: Effect of min_samples_leaf')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: Model Comparison & Kaggle Prediction (60pt)

In [ ]:
# Cell 9: Model comparison with 5-fold CV
models = {
    'DecisionTree (depth=10)': DecisionTreeClassifier(max_depth=10, random_state=SEED),
    'DecisionTree (depth=20)': DecisionTreeClassifier(max_depth=20, random_state=SEED),
    'DecisionTree (depth=None)': DecisionTreeClassifier(max_depth=None, random_state=SEED),
    'RandomForest (best)': RandomForestClassifier(n_estimators=200, min_samples_leaf=1, random_state=SEED, n_jobs=-1),
    'AdaBoost (n=100, lr=0.1)': AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED),
    'AdaBoost (n=200, lr=0.5)': AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=SEED),
    'GradientBoosting (n=200, d=5)': GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=SEED),
    'GradientBoosting (n=300, d=7)': GradientBoostingClassifier(n_estimators=300, max_depth=7, learning_rate=0.1, random_state=SEED),
}

comparison_results = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv,
                           scoring='accuracy', return_train_score=True)
    result = {
        'Model': name,
        'Train Accuracy': f"{scores['train_score'].mean():.4f} ± {scores['train_score'].std():.4f}",
        'Val Accuracy': f"{scores['test_score'].mean():.4f} ± {scores['test_score'].std():.4f}",
        'val_mean': scores['test_score'].mean()
    }
    comparison_results.append(result)
    print(f'{name:<35s}: train={scores["train_score"].mean():.4f}, val={scores["test_score"].mean():.4f}')

comp_df = pd.DataFrame(comparison_results)
comp_df[['Model', 'Train Accuracy', 'Val Accuracy']]

In [ ]:
# Cell 10: Hyperparameter tuning for GradientBoosting
param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [5, 7, 9],
    'learning_rate': [0.05, 0.1, 0.15],
    'min_samples_leaf': [1, 2, 5]
}

gb = GradientBoostingClassifier(random_state=SEED)
grid_search = GridSearchCV(
    gb, param_grid, cv=cv, scoring='accuracy',
    n_jobs=-1, verbose=1, refit=True
)
grid_search.fit(X_train, y_train)

print(f'\nBest params: {grid_search.best_params_}')
print(f'Best CV accuracy: {grid_search.best_score_:.4f}')

In [ ]:
# Cell 11: Final model training & evaluation
best_model = grid_search.best_estimator_

# Training accuracy
y_train_pred = best_model.predict(X_train)
print(f'Training accuracy: {accuracy_score(y_train, y_train_pred):.4f}')
print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV accuracy: {grid_search.best_score_:.4f}')

# Classification report
print('\nClassification Report (on training data):')
print(classification_report(y_train, y_train_pred, target_names=le.classes_))

# Confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
cm = confusion_matrix(y_train, y_train_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title('Confusion Matrix (Training Data)')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Generate Kaggle submission
y_test_pred = best_model.predict(X_test)
y_test_labels = le.inverse_transform(y_test_pred)

submission = pd.DataFrame({
    'total': test_df['total'],
    'predicted_label': y_test_labels
})

# Verify format matches sample submission
print('Submission shape:', submission.shape)
print('Sample submission shape:', sample_sub.shape)
print('Columns match:', list(submission.columns) == list(sample_sub.columns))
print('\nPredicted genre distribution:')
print(submission['predicted_label'].value_counts())

submission.to_csv('submission.csv', index=False)
print('\nSubmission saved to submission.csv')
submission.head(10)